In [1]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

In [2]:
def create_data_model():
    data = {}
    data["time_matrix"] = [
        [0, 6, 9, 8, 7, 3, 6, 2, 3, 2, 6, 6, 4, 4, 5, 9, 7],
        [6, 0, 8, 3, 2, 6, 8, 4, 8, 8, 13, 7, 5, 8, 12, 10, 14],
        [9, 8, 0, 11, 10, 6, 3, 9, 5, 8, 4, 15, 14, 13, 9, 18, 9],
        [8, 3, 11, 0, 1, 7, 10, 6, 10, 10, 14, 6, 7, 9, 14, 6, 16],
        [7, 2, 10, 1, 0, 6, 9, 4, 8, 9, 13, 4, 6, 8, 12, 8, 14],
        [3, 6, 6, 7, 6, 0, 2, 3, 2, 2, 7, 9, 7, 7, 6, 12, 8],
        [6, 8, 3, 10, 9, 2, 0, 6, 2, 5, 4, 12, 10, 10, 6, 15, 5],
        [2, 4, 9, 6, 4, 3, 6, 0, 4, 4, 8, 5, 4, 3, 7, 8, 10],
        [3, 8, 5, 10, 8, 2, 2, 4, 0, 3, 4, 9, 8, 7, 3, 13, 6],
        [2, 8, 8, 10, 9, 2, 5, 4, 3, 0, 4, 6, 5, 4, 3, 9, 5],
        [6, 13, 4, 14, 13, 7, 4, 8, 4, 4, 0, 10, 9, 8, 4, 13, 4],
        [6, 7, 15, 6, 4, 9, 12, 5, 9, 6, 10, 0, 1, 3, 7, 3, 10],
        [4, 5, 14, 7, 6, 7, 10, 4, 8, 5, 9, 1, 0, 2, 6, 4, 8],
        [4, 8, 13, 9, 8, 7, 10, 3, 7, 4, 8, 3, 2, 0, 4, 5, 6],
        [5, 12, 9, 14, 12, 6, 6, 7, 3, 3, 4, 7, 6, 4, 0, 9, 2],
        [9, 10, 18, 6, 8, 12, 15, 8, 13, 9, 13, 3, 4, 5, 9, 0, 9],
        [7, 14, 9, 16, 14, 8, 5, 10, 6, 5, 4, 10, 8, 6, 2, 9, 0],
    ]
    data["time_windows"] = [
        (0, 5),  # depot
        (7, 12),  # 1
        (10, 15),  # 2
        (16, 18),  # 3
        (10, 13),  # 4
        (0, 5),  # 5
        (5, 10),  # 6
        (0, 4),  # 7
        (5, 10),  # 8
        (0, 3),  # 9
        (10, 16),  # 10
        (10, 15),  # 11
        (0, 5),  # 12
        (5, 10),  # 13
        (7, 8),  # 14
        (10, 15),  # 15
        (11, 15),  # 16
    ]
    data["num_vehicles"] = 4
    data["depot"] = 0
    return data

In [3]:
def print_solution(data, manager, routing, solution):
    print(f"Objective: {solution.ObjectiveValue()}")
    time_dimension = routing.GetDimensionOrDie("Time")
    total_time = 0
    for vehicle_id in range(data["num_vehicles"]):
        index = routing.Start(vehicle_id)
        plan_output = f"Route for Vehicle {vehicle_id} :\n"
        while not routing.IsEnd(index):
            time_var = time_dimension.CumulVar(index)
            plan_output += (
                f" {manager.IndexToNode(index)}"
                f" Time({solution.Min(time_var)},{solution.Max(time_var)})"
                " -> "
            )
            index = solution.Value(routing.NextVar(index))
        time_var = time_dimension.CumulVar(index)
        plan_output += (
            f"{manager.IndexToNode(index)}"
            f" Time({solution.Min(time_var)},{solution.Max(time_var)})\n"
        )
        plan_output += f"Time of the Route: {solution.Min(time_var)}min\n"
        print(plan_output)
        total_time += solution.Min(time_var)
    print(f"Total time of All Routes: {total_time}min")

In [4]:
def save_cumul_data(solution, routing, dimension):
    cumul_data = []
    for route_nbr in range(routing.vehicles()):
        route_data = []
        index = routing.Start(route_nbr)
        dim_var = dimension.CumulVar(index)
        route_data.append([solution.Min(dim_var), solution.Max(dim_var)])
        while not routing.IsEnd(index):
            index = solution.Value(routing.NextVar(index))
            dim_var = dimension.CumulVar(index)
            route_data.append([solution.Min(dim_var), solution.Max(dim_var)])
        cumul_data.append(route_data)
    return cumul_data

In [5]:
def save_routes(solution, routing, manager):
    routes = []
    for route_nbr in range(routing.vehicles()):
        index = routing.Start(route_nbr)
        route = [manager.IndexToNode(index)]
        while not routing.IsEnd(index):
            index = solution.Value(routing.NextVar(index))
            route.append(manager.IndexToNode(index))
        routes.append(route)
    return routes

In [6]:
def print_solution2(routes, cumul_data):
    total_time = 0
    route_str = ""
    for i, route in enumerate(routes):
        route_str += "Route " + str(i) + " :\n"
        start_time = cumul_data[i][0][0]
        end_time = cumul_data[i][0][1]
        route_str += (
            " " +
            str(route[0]) + 
            " Time(" +
            str(start_time) +
            ", " + 
            str(end_time) + 
            ")"  
        )
        for j in range(1, len(route)):
            start_time = cumul_data[i][j][0]
            end_time = cumul_data[i][j][1]
            route_str += (
                " -> " +
                str(route[j]) +
                " Time(" +
                str(start_time) +
                ", " +
                str(end_time) +
                ")"
            )
        route_str += f"\nRoute Time: {start_time}min\n\n"
        total_time += cumul_data[i][len(route)-1][0]
    route_str += f"Total Time: {total_time}min"
    print(route_str)

In [7]:
def run_vrptw():
    data = create_data_model()
    manager = pywrapcp.RoutingIndexManager(
        len(data["time_matrix"]), data["num_vehicles"], data["depot"]
    )
    routing = pywrapcp.RoutingModel(manager)

    def time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data["time_matrix"][from_node][to_node]
    transit_callback_index = routing.RegisterTransitCallback(time_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    dimension_name = "Time"
    routing.AddDimension(
        transit_callback_index,
        30,
        30,
        False,
        dimension_name
    )
    time_dimension = routing.GetDimensionOrDie(dimension_name)

    for location_idx, time_window in enumerate(data["time_windows"]):
        if location_idx == data["depot"]:
            continue
        index = manager.NodeToIndex(location_idx)
        time_dimension.CumulVar(index).SetRange(time_window[0], time_window[1])
    
    depot_idx = data["depot"]
    for vehicle_id in range(data["num_vehicles"]):
        index = routing.Start(vehicle_id)
        time_dimension.CumulVar(index).SetRange(
            data["time_windows"][depot_idx][0], data["time_windows"][depot_idx][1]
        )
    
    for i in range(data["num_vehicles"]):
        routing.AddVariableMinimizedByFinalizer(time_dimension.CumulVar(routing.Start(i)))
        routing.AddVariableMinimizedByFinalizer(time_dimension.CumulVar(routing.End(i)))
    
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()

    solution = routing.SolveWithParameters(search_parameters)
    if solution:
        print_solution(data, manager, routing, solution)
        print("\n---- Printing Part 2 ----\n")
        cumul_data = save_cumul_data(solution, routing, time_dimension)
        routes = save_routes(solution, routing, manager)
        print_solution2(routes, cumul_data)
    else:
        print("No Solution!")

In [8]:
run_vrptw()

Objective: 71
Route for Vehicle 0 :
 0 Time(0,0) ->  9 Time(2,3) ->  14 Time(7,8) ->  16 Time(11,11) -> 0 Time(18,18)
Time of the Route: 18min

Route for Vehicle 1 :
 0 Time(0,0) ->  7 Time(2,4) ->  1 Time(7,11) ->  4 Time(10,13) ->  3 Time(16,16) -> 0 Time(24,24)
Time of the Route: 24min

Route for Vehicle 2 :
 0 Time(0,0) ->  12 Time(4,4) ->  13 Time(6,6) ->  15 Time(11,11) ->  11 Time(14,14) -> 0 Time(20,20)
Time of the Route: 20min

Route for Vehicle 3 :
 0 Time(0,0) ->  5 Time(3,3) ->  8 Time(5,5) ->  6 Time(7,7) ->  2 Time(10,10) ->  10 Time(14,14) -> 0 Time(20,20)
Time of the Route: 20min

Total time of All Routes: 82min

---- Printing Part 2 ----

Route 0 :
 0 Time(0, 0) -> 9 Time(2, 3) -> 14 Time(7, 8) -> 16 Time(11, 11) -> 0 Time(18, 18)
Route Time: 18min

Route 1 :
 0 Time(0, 0) -> 7 Time(2, 4) -> 1 Time(7, 11) -> 4 Time(10, 13) -> 3 Time(16, 16) -> 0 Time(24, 24)
Route Time: 24min

Route 2 :
 0 Time(0, 0) -> 12 Time(4, 4) -> 13 Time(6, 6) -> 15 Time(11, 11) -> 11 Time(14, 1